<!-- paper-paired-role-banner -->
> 🟦 **실습 코드 · 왼쪽 창** — TODO를 먼저 직접 구현하세요. 막히면 오른쪽 정답의 같은 셀에서 필요한 한 줄만 확인합니다.

# 19. CLIP — 이미지·텍스트 공동공간 — Learning Transferable Visual Models From Natural Language Supervision

- **원 논문:** [Learning Transferable Visual Models From Natural Language Supervision](https://arxiv.org/abs/2103.00020)
- **저자 / 발표:** Alec Radford et al. · ICML (2021)
- **난이도 / 예상 시간:** 중상급 · 약 80분
- **선수 지식:** 대조학습, cosine similarity, dual encoder, zero-shot 분류

이 노트북은 논문 결과 수치를 그대로 재현하는 대규모 benchmark가 아니라,
핵심 수식과 구조가 실제로 어떻게 동작하는지 확인하는 **하드웨어 인식 미니 재현**입니다.
PyTorch 학습은 CUDA → Apple MPS → CPU 순으로 자동 선택하며, 작은 NumPy·Python 실험은
전송 비용을 피하기 위해 CPU에서 실행될 수 있습니다.

## 재현 범위와 완료 기준

**이 노트북이 재현하는 것**

그림 1과 §2.3의 image/text dual encoder, L2 정규화, 학습 가능한 temperature, 대칭 cross entropy를 합성 paired feature에 구현합니다. image→text retrieval과 class-name prompt를 이용한 zero-shot 분류 경로를 확인합니다.

**원 논문과 다른 것**

4억 개 웹 image-text pair, ResNet/ViT와 Transformer text encoder, prompt ensemble 및 30개 이상 benchmark는 생략합니다. 연속 벡터로 만든 작은 paired dataset이므로 실제 자연어·영상 일반화와 표 3~12의 수치를 재현하지 않습니다.

완료하려면 TODO를 구현하고, 검증 셀의 `assert`를 모두 통과시키고, 마지막 관찰 질문에
자신의 말로 답하세요. 수치가 논문과 다르더라도 핵심 불변식과 비교 방향이 맞으면 성공입니다.

## 논문 ↔ 노트북 지도

        절 번호와 수식 번호는 위 링크의 대표 공개본을 기준으로 합니다. 판본에 따라 페이지는 달라질 수
        있으므로 **절 제목과 수식 번호**를 함께 사용하세요.

        | 원 논문의 위치 | 이 노트북의 대응 실습 | 확인할 증거 |
        |---|---|---|
        | 그림 1 (1)~(2) | image/text encoder와 normalized embedding | [N,D] 단위벡터 |
| 그림 1 (3), §2.3 | pairwise cosine logits와 temperature | [N,N] similarity |
| 그림 1 pseudocode | image/text 양방향 symmetric loss | 두 cross entropy 평균 |
| §3.1 | image↔text retrieval | Recall@1 |
| §2.1 및 §3.1 | class-name text prototype zero-shot 분류 | held-out accuracy |

        > 읽기 순서: 먼저 해당 절을 훑고 → 코드를 예측해 작성하고 → 출력이 논문의 주장과 왜
        > 연결되는지 한 문장으로 설명합니다.

## 핵심 수식과 구현 설계

$$
s_{ij}=\frac{f_I(I_i)^\top f_T(T_j)}{\tau},\qquad
\mathcal{L}_{\mathrm{CLIP}}=
\tfrac12\bigl(\operatorname{CE}(s,y)+\operatorname{CE}(s^\top,y)\bigr)
$$

- $f_I,f_T$는 image/text encoder, $s_{ij}$는 정규화된 cross-modal 유사도입니다.
- 같은 batch index를 positive로 두고 image-to-text와 text-to-image 분류를 대칭 학습합니다.
- 두 encoder, L2 normalization, logit scale, 대칭 loss, zero-shot 비교를 분리해 구현합니다.
- embedding $[B,D]$ 두 개의 행렬곱으로 similarity matrix $[B,B]$를 만듭니다.

## 구현 선택 이유

- **원문에서 보존한 부분:** 그림 1 (1)~(2)
- **노트북 구현:** image/text encoder와 normalized embedding
- **이렇게 구현한 이유:** 완성된 고수준 model을 한 줄로 호출하면
  논문의 핵심 shape·mask·loss·상태 변화가 숨겨집니다. 따라서 이 계산을
  이름이 드러나는 class와 함수로 나누고 중간 tensor를 assertion으로
  검사합니다.
- **실패를 잡는 증거:** [N,D] 단위벡터를 출력 크기만 아니라
  구현 불변식으로 사용합니다.
- **축소 선택:** 4억 개 웹 image-text pair, ResNet/ViT와 Transformer text encoder, prompt ensemble 및 30개 이상 benchmark는 생략합니다. 연속 벡터로 만든 작은 paired dataset이므로 실제 자연어·영상 일반화와 표 3~12의 수치를 재현하지 않습니다.는 기본 실행에서 생략합니다.
  이는 Windows CPU에서도 반복 실험이 가능하게 하면서 핵심 메커니즘은
  유지하기 위한 선택입니다.

### 구현 단계 1 · 실행 환경·데이터 계약 (setup 예외)

- **원문 위치:** 해당 없음 — import, seed, device, 로컬 데이터 로딩만 담당하며 논문의 학습 연산을 구현하지 않습니다.
- **Tensor shape 계약:** image/text features [B,*] → embeddings [B,D] → similarity [B,B]
- **구현 이유:** 이후 셀의 비교가 재현 가능하도록 장치·dtype·seed와 입력 불변식을 먼저 고정합니다.
- **완료 증거:** 데이터 존재 여부, 입력 rank, 장치 선택 assertion과 환경 요약 출력이 통과해야 합니다.

In [ ]:
import math
import random

import matplotlib.pyplot as plt
import numpy as np
import torch
from torch import Tensor, nn
from torch.nn import functional as F
from llm_engineering_lab.acceleration import get_accelerator

random.seed(0)
np.random.seed(0)
torch.manual_seed(0)
ACCELERATOR = get_accelerator()
DEVICE = ACCELERATOR.device
if DEVICE.type == "cuda":
    torch.cuda.manual_seed_all(0)
print(ACCELERATOR.summary())
print("synthetic-data preprocessing stays on CPU; neural training uses DEVICE")

### 구현 단계 2 · 합성 입력·데이터 계약 (data-contract 예외)

- **원문 위치:** 해당 없음 — 원 논문의 학습 연산이 아니라 외부 다운로드를 없애기 위한 축소 입력 생성 단계입니다.
- **Tensor shape 계약:** image/text features [B,*] → embeddings [B,D] → similarity [B,B]
- **구현 이유:** 논문 메커니즘과 데이터 규모 효과를 분리하고 Windows CPU에서도 같은 입력을 재현합니다.
- **완료 증거:** pair 정렬, batch 크기, dtype과 입력 rank assertion이 통과해야 합니다.

In [ ]:
# 외부 이미지/텍스트 대신 같은 latent 의미에서 서로 다른 관측 feature를 만듭니다.
generator = torch.Generator().manual_seed(19)
class_centers = torch.tensor(
    [
        [2.0, 0.0, 0.0, 0.0],
        [-2.0, 0.0, 0.0, 0.0],
        [0.0, 2.0, 0.0, 0.0],
        [0.0, -2.0, 0.0, 0.0],
    ]
)
pair_labels = torch.arange(4).repeat_interleave(24)
latent = class_centers[pair_labels] + 0.35 * torch.randn(96, 4, generator=generator)
image_map = torch.randn(4, 12, generator=generator)
text_map = torch.randn(4, 10, generator=generator)
image_features = latent @ image_map + 0.03 * torch.randn(96, 12, generator=generator)
text_features = latent @ text_map + 0.03 * torch.randn(96, 10, generator=generator)
print(image_features.shape, text_features.shape)

## 1) 그림 1의 dual encoder와 similarity matrix

각 encoder의 출력은 L2 정규화합니다. 따라서 matrix product는 cosine similarity이고,
`exp(t)`를 곱한 `N×N` logit의 대각선이 올바른 image-text pair입니다.

### 구현 단계 3 · 핵심 연산·모델

- **원문 위치:** 그림 1 (1)~(2) / 그림 1 (3), §2.3
- **이 셀의 책임:** image/text encoder와 normalized embedding / pairwise cosine logits와 temperature
- **Tensor shape 계약:** image/text features [B,*] → embeddings [B,D] → similarity [B,B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** [N,D] 단위벡터 / [N,N] similarity. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 19-1: 두 encoder와 학습 가능한 logit scale을 구현하세요.
class TinyCLIP(nn.Module):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""

    def __init__(self, embed_dim: int = 8):
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        self._exercise_contract_pending = True

    def encode_image(self, x: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def encode_text(self, x: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result

    def logits(self, images: Tensor, texts: Tensor) -> Tensor:
        """실습 계약: 정답과 같은 공개 signature를 유지한다.
        원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
        result = None
        assert result is not None, (
            "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
        )
        return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 4 · 핵심 연산·모델

- **원문 위치:** 그림 1 pseudocode
- **이 셀의 책임:** image/text 양방향 symmetric loss
- **Tensor shape 계약:** image/text features [B,*] → embeddings [B,D] → similarity [B,B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** 두 cross entropy 평균. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 19-2: 그림 1 pseudocode처럼 image/text 방향 CE의 평균을 반환하세요.
def clip_loss(logits: Tensor) -> Tensor:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 5 · 목적함수·학습 update

- **원문 위치:** §3.1
- **이 셀의 책임:** image↔text retrieval
- **Tensor shape 계약:** image/text features [B,*] → embeddings [B,D] → similarity [B,B]
- **구현 이유:** forward 결과와 논문 objective, gradient update의 책임을 분리합니다.
- **완료 증거:** Recall@1. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 19-3: paired batch의 대각선을 정답으로 대칭 대조학습하세요.
clip_model = ACCELERATOR.move(TinyCLIP())
image_features_device, text_features_device = ACCELERATOR.move(
    image_features, text_features
)
optimizer = torch.optim.Adam(clip_model.parameters(), lr=0.008)
history = []


def retrieval_metrics(logits: Tensor) -> dict[str, float]:
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

### 구현 단계 6 · 핵심 연산·모델

- **원문 위치:** §2.1 및 §3.1
- **이 셀의 책임:** class-name text prototype zero-shot 분류
- **Tensor shape 계약:** image/text features [B,*] → embeddings [B,D] → similarity [B,B]
- **구현 이유:** 수식을 숨긴 고수준 호출 대신 핵심 변환과 shape 경계를 직접 추적합니다.
- **완료 증거:** held-out accuracy. 코드의 assertion과 출력으로 바로 확인합니다.

In [ ]:
# TODO 19-4: class 이름 prompt를 text prototype으로 보고 held-out image를 zero-shot 분류하세요.
test_labels = torch.arange(4).repeat_interleave(20)
test_latent = class_centers[test_labels] + 0.35 * torch.randn(
    80, 4, generator=generator
)
test_images = test_latent @ image_map
class_prompts = class_centers @ text_map
test_images_device, class_prompts_device, test_labels_device = ACCELERATOR.move(
    test_images, class_prompts, test_labels
)


def zero_shot_predict(model, images, class_text_features):
    """실습 계약: 정답과 같은 공개 signature를 유지한다.
    원문 위치와 반환 shape는 바로 위 Markdown 계약을 따른다."""
    result = None
    assert result is not None, "TODO: 문서의 shape 계약을 만족하는 결과를 구현하세요."
    return result


_stage_result = None
assert _stage_result is not None, "TODO: 위 단계의 결과와 검증을 구현하세요."


def _return_shape_fixture(result):
    """바로 위 Markdown의 반환 shape 계약을 확인한다."""
    assert result is not None, "TODO 결과가 아직 비어 있습니다."
    shape = getattr(result, "shape", None)
    container = isinstance(result, (tuple, list, dict, float, int))
    assert shape is not None or container
    return result

## 마무리 관찰 기록

아래 세 문장을 직접 완성하세요.

1. 이 논문의 핵심 연산은 `_____`이고, 코드에서는 `_____`에 해당한다.
2. 원 규모 실험 대신 미니 재현을 사용했기 때문에 확인할 수 없는 주장은 `_____`이다.
3. 한 가지 변수를 바꾸어 다시 실행한다면 `_____`를 바꾸고 `_____`를 측정하겠다.

**추가 실험:** seed는 유지한 채 한 변수만 바꾸고, 변경 전후의 metric을 표로 남기세요.